# Audio Source Separation Formulated as an Inverse Problem

Audio source separation is the task of decomposing an acoustic mixture into its individual constituent sound sources (such as isolating vocals, drums, bass, and other instruments from a musical master track). 

In this notebook, we formulate the source separation task from first principles using:
1. **The Forward Mixing Process & Underdetermined Physics**
2. **Ill-Posedness and the Non-Uniqueness of Solutions**
3. **The Bayesian Probabilistic Framework & Maximum A Posteriori (MAP) Estimation**
4. **Deep Generative Priors (GANs, VAEs, Normalizing Flows, and Diffusion Models)**
5. **Interactive PyTorch Simulation of MAP Source Inversion**
6. **A Comprehensive Comparative Taxonomy of Approaches**


---
## 1. The Forward Problem: Instantaneous Mixing Process

Consider an acoustic recording composed of $n$ independent physical sound sources:
$$\mathbf{s}_1, \mathbf{s}_2, \dots, \mathbf{s}_n \in \mathbb{R}^T$$
where $T$ is the number of time samples (or time-frequency bins in the short-time Fourier transform domain).

Under an **instantaneous mixing setting**, the observed mixture signal $\mathbf{x} \in \mathbb{R}^T$ is a linear combination of the underlying sources, let it be $n$ sources:
$$\mathbf{x} = \sum_{i=1}^n \alpha_i \mathbf{s}_i \tag{4.1}$$
where $\alpha_i \in \mathbb{R}$ denotes the mixing coefficient or relative gain of source $i$. For easier notation ($\alpha_i = 1$).
$$\mathbf{x} = \sum_{i=1}^n \mathbf{s}_i$$

We can stack all $n$ sources into a single concatenated state vector:
$$\mathbf{s} = \begin{bmatrix} \mathbf{s}_1 \\ \mathbf{s}_2 \\ \vdots \\ \mathbf{s}_n \end{bmatrix} \in \mathbb{R}^{nT}$$

The forward mixing operation can then be expressed as a linear operator $\mathcal{A}: \mathbb{R}^{nT} \to \mathbb{R}^T$:
$$\mathbf{x} = \mathcal{A}(\mathbf{s}) = \mathbf{A} \mathbf{s}$$
where $\mathbf{A} = \begin{bmatrix} \mathbf{I}_T & \mathbf{I}_T & \cdots & \mathbf{I}_T \end{bmatrix} \in \mathbb{R}^{T \times nT}$ is the block summation matrix, with $\mathbf{I}_T$ representing the $T \times T$ identity matrix.

---
## 2. Why Source Separation is an Ill-Posed Inverse Problem

An inverse problem seeks to recover the unknown state $\mathbf{s}$ from the observed measurement $\mathbf{x}$:
$$\text{Find } \mathbf{s} \in \mathbb{R}^{nT} \quad \text{such that} \quad \mathcal{A}(\mathbf{s}) = \mathbf{x}$$

According to **Jacques Hadamard (1902)**, a mathematical problem is well-posed if:
1. **Existence**: A solution exists for all admissible observations.
2. **Uniqueness**: The solution is unique.
3. **Stability**: The solution depends continuously on the initial data (small perturbations in $\mathbf{x}$ do not cause catastrophic changes in $\mathbf{s}$).

### 2.1 The Severe Non-Uniqueness of Source Separation
Audio source separation violates the **uniqueness** criterion catastrophically:
- The observation $\mathbf{x}$ lives in a $T$-dimensional space $\mathbb{R}^T$.
- The unknowns $\mathbf{s}$ live in an $nT$-dimensional space $\mathbb{R}^{nT}$.
- Because $n > 1$ (e.g., $n = 4$ for vocals, drums, bass, other), the forward operator $\mathbf{A}$ has a massive null space:
$$\dim(\ker(\mathbf{A})) = nT - T = (n - 1)T$$

### 2.2 The Null-Space Ambiguity
For any candidate source decomposition $\mathbf{s} = [\mathbf{s}_1, \dots, \mathbf{s}_n]^\top$ satisfying $\sum_i \mathbf{s}_i = \mathbf{x}$, let $\mathbf{v} = [\mathbf{v}_1, \dots, \mathbf{v}_n]^\top \in \ker(\mathbf{A})$ be any perturbation such that:
$$\sum_{i=1}^n \mathbf{v}_i = \mathbf{0}$$

Then the perturbed sources $\mathbf{s}' = \mathbf{s} + \mathbf{v}$ satisfy:
$$\mathcal{A}(\mathbf{s}') = \sum_{i=1}^n (\mathbf{s}_i + \mathbf{v}_i) = \sum_{i=1}^n \mathbf{s}_i + \sum_{i=1}^n \mathbf{v}_i = \mathbf{x} + \mathbf{0} = \mathbf{x}$$

**Conclusion**: Infinitely many mathematically distinct combinations of sources sum to the exact same observed mixture $\mathbf{x}$. Without additional prior assumptions about what realistic individual sources sound like, exact inversion is impossible.


---
## 3. Probabilistic Formulation & Maximum A Posteriori (MAP)

To resolve the ill-posed null-space ambiguity, we adopt a **Bayesian probabilistic framework**. In this view, we do not merely look for *any* mathematical decomposition; we seek the most plausible decomposition under statistical priors governing natural sound sources.

### 3.1 Statistical Source Priors
Different audio sources exhibit distinct physical and acoustic characteristics (e.g., periodic vocal cord vibrations, sharp percussive transients, resonant bass lines). We model each source as a random variable drawn from a specific prior distribution:
$$\mathbf{s}_i \sim p_{G_i}(\mathbf{s}_i) \tag{4.2}$$
where $p_{G_i}$ is the probability density function characterizing realistic signals for source class $i$.

### 3.2 The Mixture Likelihood
The relationship between the sources and the observed mixture is governed by a conditional probability distribution:
$$\mathbf{x} \mid \mathbf{s}_1, \dots, \mathbf{s}_n \sim p_{\text{mix}}\left(\mathbf{x} \;\Big|\; \sum_{i=1}^n \mathbf{s}_i\right) \tag{4.3}$$
where $p_{\text{mix}}$ models any observation noise, recording artifacts, or sensor imperfections $\boldsymbol{\epsilon}$:
$$\mathbf{x} = \sum_{i=1}^n \mathbf{s}_i + \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I})$$

Under this isotropic Gaussian observation model, the log-likelihood is directly proportional to the squared $L_2$ reconstruction error:
$$p_{\text{mix}}\left(\mathbf{x} \;\Big|\; \sum_{i=1}^n \mathbf{s}_i\right) = \frac{1}{(2\pi\sigma^2)^{T/2}} \exp\left(-\frac{1}{2\sigma^2} \left\| \mathbf{x} - \sum_{i=1}^n \mathbf{s}_i \right\|_2^2\right)$$

### 3.3 Bayesian Posterior Derivation
Assuming the sources are **unconditionally independent** ($p(\mathbf{s}_1, \dots, \mathbf{s}_n) = \prod_{i=1}^n p_{G_i}(\mathbf{s}_i)$), Bayes' theorem gives the joint posterior distribution:
$$p(\mathbf{s}_1, \dots, \mathbf{s}_n \mid \mathbf{x}) = \frac{p_{\text{mix}}\left(\mathbf{x} \;\Big|\; \sum_{i=1}^n \mathbf{s}_i\right) \prod_{i=1}^n p_{G_i}(\mathbf{s}_i)}{p(\mathbf{x})}$$

Since $p(\mathbf{x})$ is independent of the sources, we obtain the proportional posterior:
$$p(\mathbf{s}_1, \dots, \mathbf{s}_n \mid \mathbf{x}) \propto p_{\text{mix}}\left(\mathbf{x} \;\Big|\; \sum_{i=1}^n \mathbf{s}_i\right) \prod_{i=1}^n p_{G_i}(\mathbf{s}_i) \tag{4.4}$$

### 3.4 The MAP Inversion Objective
To extract the separated sources, we find the configuration that maximizes the posterior probability (Maximum A Posteriori):
$$\{\mathbf{s}_1^*, \dots, \mathbf{s}_n^*\} = \arg\max_{\mathbf{s}_1, \dots, \mathbf{s}_n} \log p(\mathbf{s}_1, \dots, \mathbf{s}_n \mid \mathbf{x})$$

Expanding the logarithms, we obtain the canonical energy minimization formulation:
$$\{\mathbf{s}_1^*, \dots, \mathbf{s}_n^*\} = \arg\min_{\mathbf{s}_1, \dots, \mathbf{s}_n} \left[ \underbrace{-\log p_{\text{mix}}\left(\mathbf{x} \;\Big|\; \sum_{i=1}^n \mathbf{s}_i\right)}_{\text{Data Consistency / Likelihood Term}} + \sum_{i=1}^n \underbrace{\left( -\log p_{G_i}(\mathbf{s}_i) \right)}_{\text{Source Prior Regularization Term}} \right]$$

For Gaussian measurement noise with variance $\sigma^2$:
$$\{\mathbf{s}_1^*, \dots, \mathbf{s}_n^*\} = \arg\min_{\mathbf{s}_1, \dots, \mathbf{s}_n} \left[ \frac{1}{2\sigma^2} \left\| \mathbf{x} - \sum_{i=1}^n \mathbf{s}_i \right\|_2^2 - \sum_{i=1}^n \log p_{G_i}(\mathbf{s}_i) \right]$$

### 3.5 The Power of "Source-Only Supervision"
This Bayesian inverse formulation reveals a profound paradigm advantage:
- **Supervised Discriminative Models (e.g., Demucs)** require massive datasets of aligned paired mixtures and ground-truth isolated stems $(\mathbf{x}, \{\mathbf{s}_i\})$, which are expensive, scarce, and limited to predefined mixing recipes.
- **Generative Inverse Formulation** is **source-only supervised**. The prior distributions $p_{G_i}$ are learned using *only isolated single-source audio tracks* (e.g., solo vocal recordings, solo drum tracks). The mixing physics is handled explicitly by the forward operator during inference!


---
## 4. Deep Generative Priors: From Manifolds to Normalizing Flows

The critical challenge in solving the MAP optimization is: **how do we parameterize and evaluate the high-dimensional prior $\log p_{G_i}(\mathbf{s}_i)$?**

### 4.1 Low-Dimensional Manifold Priors (GANs and VAEs)
Early deep priors (such as Bora et al., 2017) modeled natural signals as the output of a deterministic generator $G_\theta: \mathbb{R}^d \to \mathbb{R}^T$ mapping from a low-dimensional latent space $\mathbf{z} \in \mathbb{R}^d$ ($d \ll T$):
$$\mathbf{s} = G_\theta(\mathbf{z}), \quad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

Under this model, the inverse problem reduces to optimizing over the compact latent vectors:
$$\{\mathbf{z}_1^*, \dots, \mathbf{z}_n^*\} = \arg\min_{\mathbf{z}_1, \dots, \mathbf{z}_n} \left\| \mathbf{x} - \sum_{i=1}^n G_{\theta_i}(\mathbf{z}_i) \right\|_2^2 + \lambda \sum_{i=1}^n \|\mathbf{z}_i\|_2^2$$

#### The Fundamental Limitation: Representation Error
GANs and VAEs restrict solutions strictly to the range of the generator:
$$\text{Range}(G_\theta) = \{ G_\theta(\mathbf{z}) \mid \mathbf{z} \in \mathbb{R}^d \} \subset \mathbb{R}^T$$
Because natural audio has subtle, complex acoustic details that might not lie exactly on the low-dimensional learned manifold, GAN priors suffer from **representation error**:
$$\min_{\mathbf{z}} \| \mathbf{s}_{\text{true}} - G_\theta(\mathbf{z}) \|_2^2 > 0$$
If an authentic source has features slightly out of the training distribution, the GAN *cannot* represent it, leading to audible distortion, phase artifacts, and severe bias.

---

### 4.2 Normalizing Flows: Invertible Networks with Zero Representation Error
A **Normalizing Flow** is an exact generative model that transforms a simple base distribution $p_{\mathbf{z}}(\mathbf{z})$ (typically a standard multivariate Gaussian $\mathcal{N}(\mathbf{0}, \mathbf{I})$) into a complex target distribution $p_{\mathbf{y}}(\mathbf{y})$ through an **invertible, differentiable mapping** $f_\theta$:
$$\mathbf{y} = f_\theta(\mathbf{z}) \quad \Longleftrightarrow \quad \mathbf{z} = f_\theta^{-1}(\mathbf{y})$$

By the multivariate **change of variables formula**, the exact log probability density of $\mathbf{y}$ is:
$$\log p_{\mathbf{y}}(\mathbf{y}) = \log p_{\mathbf{z}}(f_\theta^{-1}(\mathbf{y})) - \log \left| \det \frac{\partial f_\theta(\mathbf{z})}{\partial \mathbf{z}} \right| \tag{4.5}$$
or equivalently, in terms of the inverse mapping $g_\theta = f_\theta^{-1}$:
$$\log p_{\mathbf{y}}(\mathbf{y}) = \log p_{\mathbf{z}}(g_\theta(\mathbf{y})) + \log \left| \det \frac{\partial g_\theta(\mathbf{y})}{\partial \mathbf{y}} \right|$$

#### Key Architectures for Tractable Jacobians:
To make Eq. (4.5) computationally tractable for high-dimensional signals:
1. **NICE (Non-linear Independent Components Estimation)**:
   Splits input $\mathbf{x} = [\mathbf{x}_1, \mathbf{x}_2]$ and applies additive coupling layers:
   $$\mathbf{y}_1 = \mathbf{x}_1, \quad \mathbf{y}_2 = \mathbf{x}_2 + m(\mathbf{x}_1)$$
   The Jacobian is unit triangular, so $\det J = 1$ (volume-preserving) and $\log \det J = 0$.
2. **RealNVP (Real-valued Non-Volume Preserving)**:
   Employs affine coupling layers with scaling and translation:
   $$\mathbf{y}_1 = \mathbf{x}_1, \quad \mathbf{y}_2 = \mathbf{x}_2 \odot \exp(s(\mathbf{x}_1)) + t(\mathbf{x}_1)$$
   The Jacobian is triangular, and the determinant is simply the sum of scaling factors: $\log \det J = \sum_j s(\mathbf{x}_1)_j$.
3. **Glow (Generative Flow with Invertible $1\times 1$ Convolutions)**:
   Adds learned invertible $1 \times 1$ convolutions and Actnorm (activation normalization) layers to allow flexible channel mixing while maintaining $O(C)$ Jacobian calculation.

#### Why Normalizing Flows are Ideal for Inverse Problems:
1. **Zero Representation Error**: Because $f_\theta$ is a bijection ($f_\theta: \mathbb{R}^T \to \mathbb{R}^T$), **every single point in $\mathbb{R}^T$ has a unique representation**. There is no dimensional bottleneck or range truncation.
2. **Out-of-Distribution (OOD) Robustness**: Flow models assign non-zero probability densities everywhere and can faithfully invert arbitrary inputs, overcoming the rigid bias of GAN manifolds.
3. **Exact MAP Gradients**: The exact log-likelihood can be differentiated directly with respect to the input source signals:
$$\nabla_{\mathbf{s}_i} \log p_{G_i}(\mathbf{s}_i) = \nabla_{\mathbf{s}_i} \left[ \log p_{\mathbf{z}}(f_{\theta_i}^{-1}(\mathbf{s}_i)) - \log \left| \det J_{f_{\theta_i}} \right| \right]$$


---
### 4.3 Modern Diffusion and Score-Based Generative Priors
In contemporary state-of-the-art audio generation (such as **Stable Audio 3**'s Diffusion Transformers / DiTs), generative models represent the signal prior through score functions:
$$\mathbf{s}_\theta(\mathbf{x}_t, t) \approx \nabla_{\mathbf{x}_t} \log p_t(\mathbf{x}_t)$$

To solve the inverse problem using a diffusion prior, **Diffusion Posterior Sampling (DPS)** guides the reverse diffusion trajectory using the gradient of the measurement consistency:
$$\nabla_{\mathbf{s}_{t}} \log p_t(\mathbf{s}_t \mid \mathbf{x}) = \underbrace{\nabla_{\mathbf{s}_{t}} \log p_t(\mathbf{s}_t)}_{\text{Unconditional Score Prior}} + \underbrace{\nabla_{\mathbf{s}_{t}} \log p_t(\mathbf{x} \mid \mathbf{s}_t)}_{\text{Measurement Consistency Guidance}}$$
This allows solving source separation by taking reverse diffusion steps guided by the constraint $\sum_i \hat{\mathbf{s}}_{0|t, i} \approx \mathbf{x}$ at each sampling step!


---
## 6. Comprehensive Taxonomy: Approaches to Solving the Inverse Problem

The table below contrasts the different paradigms for solving source separation as an inverse problem, detailing their mathematical mechanisms, supervision requirements, representation errors, and practical trade-offs:

| Paradigm / Approach | Key Representative Methods | Mathematical Mechanism | Supervision Level | Representation Error | Out-of-Distribution (OOD) Robustness | Inference Speed & Compute | Key Strengths & Failure Modes |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **1. Classical Heuristic / Sparsity** | Independent Component Analysis (ICA), Non-negative Matrix Factorization (NMF), Wiener Filtering | Handcrafted convex priors (e.g., spectral non-negativity $\mathbf{W}\mathbf{H} \approx \mathbf{X}$, $L_1$ W-disjoint orthogonality in STFT). | **Unsupervised** (requires no training data; purely mathematical optimization). | **High** (handcrafted mathematical priors cannot capture complex musical timbre). | **High** (no dataset overfitting), but baseline reconstruction quality is low. | **Fast to Moderate** (iterative matrix updates / convex optimization). | **Strengths**: Interpretable, no training data required.<br>**Weaknesses**: Severe musical noise, phase distortion, poor separation on complex polyphonic music. |
| **2. Supervised Discriminative Masking / Direct Regression** | Open-Unmix (LSTM), Conv-TasNet (TCN), Demucs v1-v4 (U-Net + BiLSTM / Conv), HT-Demucs (Hybrid Transformer) | Trains deep network $f_\theta(\mathbf{x})$ directly on paired dataset to predict complex/magnitude masks or direct waveforms: $\min_\theta \sum \| \mathbf{s}_i - f_{\theta, i}(\mathbf{x}) \|$. | **Fully Supervised Mixture-Source Pairs** (requires synchronized stems $\mathbf{x} = \sum \mathbf{s}_i$). | **N/A** (discriminative mapping; does not invert a generative model). | **Low to Moderate** (vulnerable to domain mismatch, novel instruments, or unseen mixing gains). | **Extremely Fast** (single feed-forward pass $O(1)$). | **Strengths**: State-of-the-art SDR benchmarks on in-distribution datasets (e.g., MUSDB18).<br>**Weaknesses**: Cannot separate unseen source types without retraining; requires scarce paired multitrack datasets. |
| **3. Generative Manifold Priors (GANs / VAEs)** | Bora et al. (Compressed Sensing using GANs), Audio-GAN Inversion, AmbientGAN | Low-dimensional latent search: $\min_{\mathbf{z}_1, \dots, \mathbf{z}_n} \| \mathbf{x} - \sum G_{\theta_i}(\mathbf{z}_i) \|_2^2 + \lambda \sum \|\mathbf{z}_i\|_2^2$. | **Source-Only Supervised** (trained only on clean isolated single sources). | **High** ($\min_\mathbf{z} \|\mathbf{s} - G(\mathbf{z})\| > 0$; cannot represent signals outside the generator manifold). | **Poor** (fails on acoustic details or instruments outside the training manifold; mode collapse). | **Moderate** (gradient descent in low-dimensional latent space $\mathbf{z} \in \mathbb{R}^d$). | **Strengths**: Enforces strong structural consistency; low-dimensional optimization.<br>**Weaknesses**: High representation error causes blurs, robotic artifacts, and inability to recover fine acoustic transients. |
| **4. Invertible Neural Networks / Normalizing Flows** | NICE, RealNVP, Glow, WaveGlow, Flow-based Source Separation | Exact bijective change of variables: $\log p(\mathbf{s}) = \log p(\mathbf{z}) - \log \|\det J_f\|$. Inverts via exact MAP gradient ascent on input audio: $\nabla_{\mathbf{s}_i} \log p(\mathbf{s}_i \mid \mathbf{x})$. | **Source-Only Supervised** (flows trained to maximize exact log-likelihood on single stems). | **Zero** (bijection $f: \mathbb{R}^T \to \mathbb{R}^T$ has full ambient rank; covers the entire signal space). | **Superior** (assigns valid probabilities everywhere; robust to out-of-distribution sounds). | **Moderate to Slow** (requires iterative gradient backpropagation through invertible flow chain). | **Strengths**: Exact likelihood, zero representation error, provable invertibility, excellent OOD generalization.<br>**Weaknesses**: Flow architectures require volume/Jacobian constraints, increasing parameter count. |
| **5. Diffusion & Score-Based Posterior Sampling** | Diffusion Posterior Sampling (DPS), Plug-and-Play (PnP-DM), DiffSep, Stable Audio Prior Inversion | Score-based SDE/ODE guidance: $\nabla_{\mathbf{s}_t} \log p_t(\mathbf{s}_t \mid \mathbf{x}) = \nabla_{\mathbf{s}_t} \log p_t(\mathbf{s}_t) + \nabla_{\mathbf{s}_t} \log p(\mathbf{x} \mid \hat{\mathbf{s}}_{0|t})$. | **Source-Only Supervised** (trained unconditionally or class-conditioned on single sources). | **Extremely Low** (full-dimensional expressive generative diffusion space). | **High** (stochastic diffusion process explores multiple plausible modes without mode collapse). | **Slow** (requires 25–100 iterative score evaluations / denoising steps). | **Strengths**: Phenomenal perceptual quality, handles extreme ill-posedness and underdetermined cases, supports flexible conditioning (text, duration).<br>**Weaknesses**: Computationally intensive sampling; requires careful calibration of measurement guidance scale $\zeta$. |

---

## 7. Key Takeaways & Architectural Insights

1. **The Separation Duality**:
   - **Discriminative models** learn the conditional posterior $p(\mathbf{s} \mid \mathbf{x})$ directly via regression, trading generalizability and modularity for fast, single-pass inference.
   - **Generative inverse models** factor the problem into physical forward modeling (the likelihood $p(\mathbf{x} \mid \mathbf{s})$) and independent source acoustics (the priors $p(\mathbf{s}_i)$), achieving **source-only supervision** and unmatched modularity.

2. **Why Flow and Diffusion Priors Dominate over GANs for Inversion**:
   - GANs impose a strict low-dimensional bottleneck that discards off-manifold details.
   - **Normalizing Flows** provide mathematically guaranteed **zero representation error** and exact likelihoods, making them the purest Bayesian regularizer for inverse problems.
   - **Diffusion Models** (as used in **Stable Audio 3**) provide state-of-the-art perceptual sample quality and can be leveraged through posterior sampling (DPS) to separate complex polyphonic mixtures without ever training on paired multitrack datasets.
